# 04 · Recovery and human review

**Prerequisites:** Understand graph state and branching; all prerequisite code is supplied here.

**Learning objectives:** Retry only temporary failures; checkpoint a review; resume or reject without repeating completed retrieval.

**Guide companion:** sections 8 in `LANGCHAIN_LANGGRAPH_LEARNING_GUIDE.md` at the project root.

**How to work:** Run setup, implement each challenge, then run its acceptance cell. Starter functions deliberately raise `NotImplementedError`; this is expected until you complete them. Restart the kernel and run all cells after finishing. You do not need any other notebook or paid API calls. Budget about 45–90 minutes, or longer for the capstone.

Complete solutions are kept in the matching notebook under `solutions/`. There are no hidden solution cells in this notebook.


In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

# Fictional test data, not real people, policies, or research sources.
NOTES = [
    {"id": "s1", "url": "fixture://architecture", "text": "Cedar uses LangGraph to route research tasks."},
    {"id": "s2", "url": "fixture://review", "text": "Cedar pauses its workflow for human review."},
    {"id": "s3", "url": "fixture://ownership", "text": "Mira maintains Cedar."},
    {"id": "s4", "url": "fixture://team", "text": "Mira works on team Atlas."},
    {"id": "s5", "url": "fixture://policy", "text": "Atlas reviews Cedar evidence monthly."},
]
BY_ID = {note["id"]: note for note in NOTES}

from operator import add
from typing import Annotated, TypedDict
from uuid import uuid4
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, RetryPolicy, interrupt

class TemporarySearchError(Exception):
    pass

class ReviewState(TypedDict):
    claims: list[dict]
    status: str
    approved: bool
    log: Annotated[list[str], add]

def initial():
    return {"claims": [], "status": "running", "approved": False, "log": []}

def fresh_config():
    return {"configurable": {"thread_id": str(uuid4())}, "recursion_limit": 20}

def make_fetch(failures=0, error_type=TemporarySearchError):
    # Counter deliberately lives outside state for deterministic fault injection.
    calls = {"count": 0}
    def fetch(state):
        calls["count"] += 1
        if calls["count"] <= failures:
            raise error_type("Simulated retrieval failure")
        return {"claims": [{"text": BY_ID["s3"]["text"], "source_id": "s3"}],
                "status": "evidence_checked", "log": ["retrieve"]}
    return fetch, calls


## Challenge 1 · Define a selective retry policy

Implement `search_retry_policy()` returning `RetryPolicy`: three total attempts, `initial_interval=0.01`, no jitter, and retry only `TemporarySearchError`. Other exceptions must propagate immediately.


In [ ]:
def search_retry_policy():
    raise NotImplementedError("Challenge 1: configure bounded selective retries")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
policy = search_retry_policy()
assert isinstance(policy, RetryPolicy)
assert policy.max_attempts == 3 and policy.initial_interval == 0.01
assert policy.jitter is False and policy.retry_on is TemporarySearchError
print("PASS: policy configuration; runtime behavior is checked after graph assembly")


## Challenge 2 · Pause for a review decision

Implement `review_node(state)`: call `interrupt` with a JSON-serializable dictionary containing a review question and the claims. Accept only the exact resume value `"approve"`. Return `approved` or `rejected` status, the Boolean `approved`, and one `review` log entry. Rejection clears claims.
Do not use `input()`, write to external systems, or catch the interrupt.


In [ ]:
def review_node(state):
    raise NotImplementedError("Challenge 2: interrupt for approval and handle its resume value")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
# Exercise the node inside the runtime: interrupt is not an ordinary standalone call.
probe = StateGraph(ReviewState)
probe.add_node("review", review_node)
probe.add_edge(START, "review")
probe.add_edge("review", END)
probe_graph = probe.compile(checkpointer=InMemorySaver())
probe_config = fresh_config()
probe_input = initial()
probe_input["claims"] = [{"text": BY_ID["s3"]["text"], "source_id": "s3"}]
paused_probe = probe_graph.invoke(probe_input, probe_config)
assert "__interrupt__" in paused_probe
assert paused_probe["__interrupt__"][0].value["claims"] == probe_input["claims"]
rejected_probe = probe_graph.invoke(Command(resume="reject"), probe_config)
assert rejected_probe["status"] == "rejected" and not rejected_probe["claims"]
print("PASS: pause and reject behavior")


## Challenge 3 · Assemble a recoverable graph

Implement `build_review_graph(fetch, saver)`. Connect START → `retrieve` → `review` → END. Use the supplied fetch node, your review node, your retry policy on retrieval only, and the provided checkpointer.
Return a compiled graph. Do not silently replace `saver`; callers must be able to rebuild a graph against the same saved state.


In [ ]:
def build_review_graph(fetch, saver):
    raise NotImplementedError("Challenge 3: connect retrieval and checkpointed review")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
fetch, calls = make_fetch(failures=2)
saver = InMemorySaver()
graph = build_review_graph(fetch, saver)
config = fresh_config()
paused = graph.invoke(initial(), config)
assert calls["count"] == 3 and "__interrupt__" in paused
assert graph.get_state(config).next == ("review",)
rebuilt = build_review_graph(fetch, saver)
approved = rebuilt.invoke(Command(resume="approve"), config)
assert approved["status"] == "approved" and approved["approved"]
assert calls["count"] == 3, "Resume must not repeat completed retrieval"
assert approved["log"] == ["retrieve", "review"]

for failures, error_type, expected_attempts in [(5, TemporarySearchError, 3), (1, ValueError, 1)]:
    failing_fetch, counter = make_fetch(failures, error_type)
    try:
        build_review_graph(failing_fetch, InMemorySaver()).invoke(initial(), fresh_config())
    except error_type:
        assert counter["count"] == expected_attempts, "Incorrect retry classification or limit"
    else:
        raise AssertionError("An exhausted or non-retryable failure must remain visible")
print("PASS: retry success, exhaustion, non-retryable errors, and checkpoint resumption")


## Your turn · Pause, inspect, then decide

Run the next cell first. Inspect the review payload before running the separate resume cell. Every execution of the pause cell creates a new thread. To repeat the exercise, run the pause cell again before resuming.


In [ ]:
practice_fetch, practice_calls = make_fetch()
practice_graph = build_review_graph(practice_fetch, InMemorySaver())
practice_config = fresh_config()
practice_pause = practice_graph.invoke(initial(), practice_config)
print(practice_pause["__interrupt__"][0].value)
print("Pending:", practice_graph.get_state(practice_config).next)


Choose `"approve"` or `"reject"` below. Nothing is sent or published; this is a local review simulation.


In [ ]:
decision = "reject"
if decision not in {"approve", "reject"}:
    raise ValueError("Choose approve or reject")
if not practice_graph.get_state(practice_config).next:
    raise RuntimeError("This review is finished. Run the pause cell again to start another.")
practice_result = practice_graph.invoke(Command(resume=decision), practice_config)
assert practice_calls["count"] == 1
print(practice_result["status"], practice_result["claims"])


## Reflection

1. Which code may execute again when a review resumes?
2. Why would sending an email before `interrupt()` be dangerous?
3. What must change to recover after restarting Python?


**Your answers:**

Write your reasoning here before opening the solutions.


## References

- [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [Graph API examples](https://docs.langchain.com/oss/python/langgraph/use-graph-api)
